**Cell 1: Environment Setup & Data Extraction**

Extraction of the data from our drive

In [9]:
import os
from google.colab import drive
import zipfile

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Image Segmentation Project/Project/Task 8/Anomaly_Validation_Datasets.zip'
EXTRACT_DIR = '/content/datasets'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    print("Unzipping datasets... this might take a minute.")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Unzip complete!")
else:
    print("Datasets already extracted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets already extracted.


**Cell 2: The Reusable Metric Engine**

Definition of the metrics : output score AuPrc and FPR95

In [10]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import auc, roc_curve, precision_recall_curve

def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    """
    Calculates AuPRC and FPR95 for anomaly segmentation.
    y_true: Ground truth numpy array (0 = Normal, 1 = Anomaly, 255 = Ignore)
    y_scores: Anomaly score numpy array (Higher = more anomalous)
    """
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()

    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]

    # Force binarization: 0 stays 0, anything else (like 2) becomes 1
    y_true_filtered = (y_true_filtered > 0).astype(int)

    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0

    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]

    return auprc, fpr[idx]

**Cell 3: Anomaly Score Extractors**

- `get_msp_score`, `get_maxlogit_score`, `get_max_entropy_score` : Don't change from task 7

- `convert_eomt_output_to_pixel_logits` : **New function from task 8** .
Eomt uses N masks an N vectors instead of logits pixel, this function constructs the logits pixel-levels with the masks and vector classes.
  

- `get_rba_score` : **New FUNCTION**
  calculate the Rba score especially designed for mask architectures only

In [11]:

def get_msp_score(logits):
    """ MSP Score: 1 - max(softmax(logits)) """
    probs = F.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    anomaly_score = 1.0 - max_probs
    return anomaly_score.cpu().numpy()

def get_maxlogit_score(logits):
    """ MaxLogit Score: -max(logits) """
    max_logits, _ = torch.max(logits, dim=1)
    anomaly_score = -max_logits
    return anomaly_score.cpu().numpy()

def get_max_entropy_score(logits):
    """ Max Entropy Score: H = -sum(p * log(p)), normalisée par log(C) """
    probs = F.softmax(logits, dim=1)
    raw_entropy = torch.sum(-probs * torch.log(probs + 1e-8), dim=1)
    num_classes = probs.shape[1]
    normalized_entropy = torch.div(raw_entropy, torch.log(torch.tensor(num_classes, dtype=torch.float32, device=logits.device)))
    return normalized_entropy.cpu().numpy()



def convert_eomt_output_to_pixel_logits(outputs):
  """Convertit la sortie EoMT en logits pixel-level [B, C, H, W].
    Returns:
        pixel_logits : Tensor [B, C, H, W] utilisable par get_msp_score, etc.
    """
  # outputs[0][-1] : masques du dernier bloc [B, 100, H, W]
  # outputs[1][-1] :logits du dernier bloc   [B, 100, 20]
  mask_logits  = outputs[0][-1]  # [B, N, H, W]
  class_logits = outputs[1][-1]  # [B, N, C]  — pas de classe "no-object" ici (20 = 19+1 void)

  # Retirer la dernière classe void (index -1)
  class_logits_no_void = class_logits[:, :, :-1]  # [B, N, 19]

  class_probs = F.softmax(class_logits_no_void, dim=-1)  # [B, N, 19]
  masks = torch.sigmoid(mask_logits)

  pixel_logits = torch.einsum('bnc,bnhw->bchw', class_probs, masks)  # [B, 19, H, W]
  return pixel_logits




def get_rba_score(outputs):
    #RbA Score : 1 - max_query( sigmoid(mask) * max_class_prob )
    mask_logits  = outputs[0][-1]
    class_logits = outputs[1][-1]

    # Retirer la classe "no-object" avant softmax
    class_logits_no_void = class_logits[:, :, :-1]
    class_probs = F.softmax(class_logits_no_void, dim=-1)
    max_class_prob, _ = class_probs.max(dim=-1)

    masks = torch.sigmoid(mask_logits)  # proba spatiale de chaque mask
    claim = masks * max_class_prob.unsqueeze(-1).unsqueeze(-1)  # pixel score
    # On broadcast max_class_prob [B, N] -> [B, N, H, W]
    # Max sur toutes les queries : le score de la query la plus confiante
    max_claim, _ = claim.max(dim=1)

    return (1.0 - max_claim).cpu().numpy()

**Cell 4: Imports, Dataset, et Chargement du Modèle**

- `SHARED_FOLDER_NAME` : new folder 8
- Import the model Eomt
- `input_transform_fn` : new transform for ImageNet (needed by EoMT / ViT backbone)
- Charging the model: EoMT
- `CHECKPOINT_NAME` : variable for checkpoints

In [12]:
import sys, os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import auc, roc_curve, precision_recall_curve


SHARED_FOLDER_NAME = 'Image Segmentation Project/Project/Task 8'


EOMT_REPO_PATH = '/content/drive/MyDrive/Image Segmentation Project/Project/MaskArchitectureAnomaly_CourseProject/eomt'
sys.path.append(EOMT_REPO_PATH)


from models.eomt import EoMT
from models.transform import ToLabel
from models.vit import ViT

# ======================================================
# 3 checkpoints :
#   - 'eomt_coco.pth'
#   - 'eomt_cityscapes.pth'
#   - 'eomt_finetuned.pth'   : finetuned task 5
# Change this and reload cell 5 and afer.
# =======================================================
CHECKPOINT_NAME = 'eomt_cityscapes.bin'



def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
  #same as task
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()
    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]
    y_true_filtered = (y_true_filtered > 0).astype(int)
    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0
    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)
    fpr, tpr, _ = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]
    return auprc, fpr[idx]

class CustomAnomalyDataset(Dataset):
    def __init__(self, root, input_transform=None, target_transform=None):
        self.images_dir = os.path.join(root, 'images')
        self.masks_dir = os.path.join(root, 'labels_masks')
        image_files = sorted([f for f in os.listdir(self.images_dir) if not f.startswith('.')])
        self.samples = []
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_files = [m for m in os.listdir(self.masks_dir) if m.startswith(stem + '.')]
            if mask_files:
                self.samples.append((img_file, mask_files[0]))
        self.input_transform = input_transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        img_name, mask_name = self.samples[index]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        with open(img_path, 'rb') as f: image = Image.open(f).convert('RGB')
        with open(mask_path, 'rb') as f: mask = Image.open(f).convert('P')
        if self.input_transform: image = self.input_transform(image)
        if self.target_transform: mask = self.target_transform(mask)
        return image, mask, img_name, mask_name

    def __len__(self):
        return len(self.samples)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 1


#Charging the model with the parameters needed by each checkpoints

print(f"Loading EoMT with checkpoint: {CHECKPOINT_NAME}...")


if CHECKPOINT_NAME == 'eomt_cityscapes.bin':
    img_size    = 1024
    num_q       = 100
    num_classes = 19
    patch_size  = 16
elif CHECKPOINT_NAME == 'eomt_finetuned.bin':
    img_size    = 640
    num_q       = 200
    num_classes = 19
    patch_size  = 16
else:  # eomt_coco.bin
    img_size    = 640
    num_q       = 200
    num_classes = 133
    patch_size  = 16

encoder = ViT(
    backbone_name='vit_base_patch14_reg4_dinov2',
    img_size=(img_size, img_size),
    patch_size=patch_size,
    ckpt_path=None
)

model = EoMT(
    encoder=encoder,
    num_classes=num_classes,
    num_q=num_q,
    num_blocks=3,             # obtained from the YAML file
    masked_attn_enabled=True
).to(device)

weights_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/{CHECKPOINT_NAME}'
checkpoint = torch.load(weights_path, map_location=device)

# part to extract the right keys and transform them if needed
if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

#We had problems with key loading and weights loading, this part is to insure thatall tghe keys are here
state_dict = {k.replace('network.', ''): v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys  : {len(missing)} → {missing[:5] if missing else 'aucun'}")
print(f"Unexpected keys: {len(unexpected)} → {unexpected[:5] if unexpected else 'aucun'}")

model.eval()
print("Weights loaded successfully!")



#We normalize with mean/std ImageNet + Resize à img_size for the Vit

input_transform_fn = transforms.Compose([
    transforms.Resize((img_size, img_size)),  #  img_size from the checkpoint
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # mean ImageNet
        std=[0.229, 0.224, 0.225]    # std ImageNet
    )
])

target_transform_fn = ToLabel()

Loading EoMT with checkpoint: eomt_cityscapes.bin...


Missing keys  : 0 → aucun
Unexpected keys: 1 → ['criterion.empty_weight']
Weights loaded successfully!


**Cell 5: Boucle d'évaluation principale**

Modification from task 7
-  `convert_eomt_output_to_pixel_logits`convert the output of EoMT
- `outputs` saved for `get_rba_score` that needs the masks
- add `all_rba` and do `rba_auprc` / `rba_fpr95`
- add the rba to the return`return`  

In [14]:
def evaluate_single_dataset(dataset_name, base_dir='/content/datasets/Validation_Dataset'):
    print(f"\n--- Evaluating on {dataset_name} (checkpoint: {CHECKPOINT_NAME}) ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    #add rba scores
    all_gt, all_msp, all_maxlogit, all_entropy, all_rba = [], [], [], [], []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        masks = masks.squeeze(1).numpy()



        # I had RAM problems when i was out of collab credits so i added a resize to be still able to run the codes on the cpu
        if not torch.cuda.is_available():
          import cv2
          masks = np.stack([
                cv2.resize(m, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
                for m in masks
          ])
          h_orig, w_orig = img_size, img_size
        else:
          h_orig, w_orig = masks.shape[-2], masks.shape[-1]
        #call the model and convert the masks output into the logits for the evaluations
        outputs = model(images)

        logits = convert_eomt_output_to_pixel_logits(outputs)  # [B, C, H, W]


        logits = F.interpolate(logits, size=(h_orig, w_orig), mode='bilinear', align_corners=False)


        #resize the mask to have the same resolution as the grounf truth
        rba_masks_resized = F.interpolate(
            outputs[0][-1], size=(h_orig, w_orig), mode='bilinear', align_corners=False
        )  # [B, N, img_size, img_size]
        class_logits_no_void = outputs[1][-1][:, :, :-1]           # [B, N, C] — retirer classe void
        class_probs = F.softmax(class_logits_no_void, dim=-1)       # [B, N, C]
        max_class_prob, _ = class_probs.max(dim=-1)                 # [B, N]
        masks_sig = torch.sigmoid(rba_masks_resized)                # [B, N, img_size, img_size]
        claim = masks_sig * max_class_prob.unsqueeze(-1).unsqueeze(-1)  # [B, N, img_size, img_size]
        max_claim, _ = claim.max(dim=1)                             # [B, img_size, img_size]
        rba_score = (1.0 - max_claim).cpu().numpy()                 # anomal si aucune query ne revendique

        all_gt.append(masks.astype(np.uint8))
        all_msp.append(get_msp_score(logits).astype(np.float16))
        all_maxlogit.append(get_maxlogit_score(logits).astype(np.float16))
        all_entropy.append(get_max_entropy_score(logits).astype(np.float16))
        # score rba added
        all_rba.append(rba_score.astype(np.float16))

        #free the memory
        del images, outputs, logits, rba_masks_resized, masks_sig, claim, max_claim, rba_score
        torch.cuda.empty_cache()

    print(f"\nCalculating metrics for {dataset_name}...")
    #metrics calculation
    gt_concat = np.concatenate(all_gt)
    del all_gt

    msp_concat = np.concatenate(all_msp)
    del all_msp
    msp_auprc, msp_fpr95 = calculate_anomaly_metrics(gt_concat, msp_concat)
    del msp_concat

    maxlogit_concat = np.concatenate(all_maxlogit)
    del all_maxlogit
    ml_auprc, ml_fpr95 = calculate_anomaly_metrics(gt_concat, maxlogit_concat)
    del maxlogit_concat

    entropy_concat = np.concatenate(all_entropy)
    del all_entropy
    ent_auprc, ent_fpr95 = calculate_anomaly_metrics(gt_concat, entropy_concat)
    del entropy_concat


    rba_concat = np.concatenate(all_rba)
    del all_rba
    rba_auprc, rba_fpr95 = calculate_anomaly_metrics(gt_concat, rba_concat)
    del rba_concat

    del gt_concat

    #dictionnary of results
    return {
        "MSP":         {"AuPRC": msp_auprc, "FPR95": msp_fpr95},
        "MaxLogit":    {"AuPRC": ml_auprc,  "FPR95": ml_fpr95},
        "Max Entropy": {"AuPRC": ent_auprc, "FPR95": ent_fpr95},
        "RbA":         {"AuPRC": rba_auprc, "FPR95": rba_fpr95},
    }

**Cell 6: Lancement de l'évaluation et affichage des résultats**
- Run the functions and save the results

- Csv document saved as `Task8_Results_{CHECKPOINT_NAME}.csv`.

To change the checkpoints re run cell 4 and 6 with the new checkpoint name

In [15]:
import pandas as pd

if 'results_table' not in locals():
    results_table = {}

datasets_to_run = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

with torch.no_grad():
    for d_name in datasets_to_run:
        #save the results of the checkpoints without deleting the results of the precedent one, and save all the dico on the same csv
        results_table[f"{CHECKPOINT_NAME}_{d_name}"] = evaluate_single_dataset(d_name)

df_results = pd.DataFrame(results_table).round(4)

print(f"\nRisultati Finali — EoMT checkpoint: {CHECKPOINT_NAME}")
print(df_results.to_string(float_format='{:.3f}'.format))

csv_name = f'Task8_Results_{CHECKPOINT_NAME.replace(".bin", "")}.csv'
csv_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/{csv_name}'
df_results.to_csv(csv_path)
print(f"\nTabella salvata in: {csv_path}")


--- Evaluating on RoadAnomaly21 (checkpoint: eomt_cityscapes.bin) ---


Processing: 100%|██████████| 10/10 [00:07<00:00,  1.29it/s]



Calculating metrics for RoadAnomaly21...

--- Evaluating on RoadObsticle21 (checkpoint: eomt_cityscapes.bin) ---


Processing: 100%|██████████| 30/30 [00:27<00:00,  1.07it/s]



Calculating metrics for RoadObsticle21...

--- Evaluating on FS_LostFound_full (checkpoint: eomt_cityscapes.bin) ---


...


Calculating metrics for FS_LostFound_full...

--- Evaluating on fs_static (checkpoint: eomt_cityscapes.bin) ---


Processing: 100%|██████████| 30/30 [00:26<00:00,  1.14it/s]



Calculating metrics for fs_static...

--- Evaluating on RoadAnomaly (checkpoint: eomt_cityscapes.bin) ---


Processing: 100%|██████████| 60/60 [00:48<00:00,  1.25it/s]



Calculating metrics for RoadAnomaly...

Risultati Finali — EoMT checkpoint: eomt_cityscapes.bin
                                       eomt_cityscapes.bin_RoadAnomaly21                            eomt_cityscapes.bin_RoadObsticle21                          eomt_cityscapes.bin_FS_LostFound_full                                 eomt_cityscapes.bin_fs_static                              eomt_cityscapes.bin_RoadAnomaly
MSP          {'AuPRC': 0.36799732637481575, 'FPR95': 0.7577592065484038}   {'AuPRC': 0.30626892784017135, 'FPR95': 0.9370783364745524}     {'AuPRC': 0.0766617715091988, 'FPR95': 0.8012743150694915}   {'AuPRC': 0.22123948804111554, 'FPR95': 0.9408355073540415}   {'AuPRC': 0.3165918559554942, 'FPR95': 0.7561412695354929}
MaxLogit     {'AuPRC': 0.20063252788236796, 'FPR95': 0.7563777129412261}  {'AuPRC': 0.021633836397377294, 'FPR95': 0.9437553941998841}  {'AuPRC': 0.0033345917580160957, 'FPR95': 0.7931365467516165}  {'AuPRC': 0.012750055923158992, 'FPR95': 0.9412833105792358}  

MloU Part

In [16]:
#do not run for COCO

from torchvision.datasets import Cityscapes
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image
import torch, time
import numpy as np
#not working for coco
sys.path.insert(0, '/content/drive/MyDrive/Image Segmentation Project/Project/MaskArchitectureAnomaly_CourseProject/eval')
from iouEval import iouEval

CS_ROOT = "/content/drive/MyDrive/Image Segmentation Project/Project/2. Cityscapes"
NUM_CLASSES = 20  # 19 classes + 1 void
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


input_transform = Compose([
    Resize((img_size, img_size), Image.BILINEAR),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cs_val = Cityscapes(CS_ROOT, split='val', mode='fine',
                    target_type='semantic',
                    transform=input_transform)

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    labels = torch.stack([
        torch.from_numpy(np.array(
            item[1].resize((640, 640), Image.NEAREST),  # MODIFIÉ : 640x640
            dtype=np.int64
        ))
        for item in batch
    ])
    return images, labels

loader = torch.utils.data.DataLoader(cs_val, batch_size=1,
                                      shuffle=False, num_workers=2,
                                      collate_fn=collate_fn)

iou_eval = iouEval(NUM_CLASSES)
start = time.time()

with torch.no_grad():
    for step, (images, labels) in enumerate(loader):
        images = images.to(device)


        labels_train = torch.full_like(labels, 19, dtype=torch.long)
        for cls in Cityscapes.classes:
            if not cls.ignore_in_eval:
                labels_train[labels == cls.id] = cls.train_id
        labels_train = labels_train.to(device)


        outputs = model(images)

        #convert the outputs eomt (masks) into logits
        logits = convert_eomt_output_to_pixel_logits(outputs)  # [B, C, H, W]

        logits = F.interpolate(logits, size=(640, 640), mode='bilinear', align_corners=False)

        iou_eval.addBatch(logits.max(1)[1].unsqueeze(1).data,
                          labels_train.unsqueeze(1))

        if step % 100 == 0:
            print(f"[{step}/500] {time.time()-start:.0f}s")

iou_val, _ = iou_eval.getIoU()
print(f"\nEoMT ({CHECKPOINT_NAME}) mIoU: {iou_val*100:.1f}%")

[0/500] 7s
[100/500] 89s
[200/500] 164s
[300/500] 238s
[400/500] 312s

EoMT (eomt_cityscapes.bin) mIoU: 7.1%


**Temperature Part**

In [17]:
def evaluate_temperature_scaling(temperatures=[0.5, 0.75, 1.1],
                                  base_dir='/content/datasets/Validation_Dataset'):
    # We run the temperature only on roadanomaly and compare it for the checkpoints
    dataset_name = "RoadAnomaly21"
    print(f"\n--- Temperature Scaling on {dataset_name} (checkpoint: {CHECKPOINT_NAME}) ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    all_gt = []
    all_scores = {f't={T}': [] for T in temperatures}
    all_scores['MSP'] = []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        masks = masks.squeeze(1).numpy()

        # resize if RAM problem
        if not torch.cuda.is_available():
            import cv2
            masks = np.stack([
                cv2.resize(m, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
                for m in masks
            ])
            h_orig, w_orig = img_size, img_size
        else:
            h_orig, w_orig = masks.shape[-2], masks.shape[-1]

        outputs = model(images)
        logits = convert_eomt_output_to_pixel_logits(outputs)
        logits = F.interpolate(logits, size=(h_orig, w_orig), mode='bilinear', align_corners=False)

        # MSP de base (t=1.0)
        all_scores['MSP'].append(get_msp_score(logits).astype(np.float16))

        # MSP with each temperature, we rerun the logits
        for T in temperatures:
            logits_t = logits / T
            all_scores[f't={T}'].append(get_msp_score(logits_t).astype(np.float16))

        all_gt.append(masks.astype(np.uint8))

        del images, outputs, logits
        torch.cuda.empty_cache()

    # calcul metrics
    gt_concat = np.concatenate(all_gt)
    del all_gt

    results = {}
    for key, scores_list in all_scores.items():
        scores_concat = np.concatenate(scores_list)
        auprc, fpr95 = calculate_anomaly_metrics(gt_concat, scores_concat)
        results[key] = {"AuPRC": round(auprc, 4), "FPR95": round(fpr95, 4)}
        del scores_concat

    del gt_concat
    return results


# runing the functions
temperatures = [0.5, 0.75, 1.1]

CHECKPOINT_NAME = CHECKPOINT_NAME

with torch.no_grad():
    temp_results = evaluate_temperature_scaling(temperatures)

df_temp = pd.DataFrame(temp_results).T  #transposition to have the temperature in line
df_temp.index.name = 'Method'
print(f"\nRésultats Temperature Scaling — {CHECKPOINT_NAME} — RoadAnomaly21 :")
print(df_temp.to_string())

# Sauvegarde CSV
csv_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/Task8_Temperature_Results_{CHECKPOINT_NAME.replace(".bin", "")}.csv'
df_temp.to_csv(csv_path)
print(f"\nSauvegardé : {csv_path}")


--- Temperature Scaling on RoadAnomaly21 (checkpoint: eomt_cityscapes.bin) ---


Processing: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s]



Résultats Temperature Scaling — eomt_cityscapes.bin — RoadAnomaly21 :
         AuPRC   FPR95
Method                
t=0.5   0.3417  0.7576
t=0.75  0.3570  0.7579
t=1.1   0.3717  0.7575
MSP     0.3680  0.7578

Sauvegardé : /content/drive/MyDrive/Image Segmentation Project/Project/Task 8/Task8_Temperature_Results_eomt_cityscapes.csv
